# NHLA 1-Side Classification

This notebook trains the single-face NHLA grading model. The workflow is intentionally:
1. prepare the extracted face CSV,
2. run Optuna tuning on a short validation pass,
3. train the final 1-side model using the best hyperparameters,
4. save metrics and checkpoints for comparison against the 2-side model.

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "26-8-18_75VAL"
# Validated faces: only boards whose grader validation grade equals the vision board grade,
# labelled with the per-face vision grade (the grader never validates individual faces).
USE_VALIDATED_FACES = True
CSV_NAME = "nhla_face_labels_validated.csv" if USE_VALIDATED_FACES else "nhla_face_labels.csv"
CSV_PATH = PROJECT_ROOT / "data" / DATASET_NAME / "nhla_dataset" / CSV_NAME
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "models" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"
OPTUNA_DIR = PROJECT_ROOT / "logs" / "optuna"
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "tensorboard" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"

# ------------------------------------------------------------------
# Configuration - change these knobs as needed
# ------------------------------------------------------------------
# Input geometry (keep identical to the 2-side notebook so both share one cache and the
# 1-side model_best.pt can initialise the 2-side backbone).
IMAGE_SCALE = 0.375
CANVAS = (512, 1888)
BATCH_SIZE = 16
NUM_WORKERS = 6
NUM_EPOCHS = 10           # resnet18: ~50 min/epoch on all 582k validated faces at this canvas
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
SEED = 42
CHECKPOINT_INTERVAL = 2

# Shared decode-once image cache (same cache as the 2-side notebook). None -> ~/.cache/nhla_images
IMAGE_CACHE_DIR = None

# Optuna tuning controls
OPTUNA_ENABLED = True
OPTUNA_TRIALS = 10
TUNE_EPOCHS = 4
TUNE_SUBSET_FRACTION = 0.1  # label-stratified fraction of boards per trial (~58k faces, ~5 min/epoch)
STUDY_NAME = f"nhla_1side_optuna_{RUN_TIMESTAMP}"
OPTUNA_DB_PATH = OPTUNA_DIR / "NHLA_Classification.db"
OPTUNA_SEARCH_SPACE = {
    "lr": {"low": 1e-5, "high": 1e-3, "log": True},
    "batch_size": [16, 32],
    "weight_decay": {"low": 1e-6, "high": 1e-2, "log": True},
    "encoder_name": ["resnet18", "resnet34"],  # resnet50 is ~2.3 h/epoch at this canvas
}

# Run naming
CHECKPOINT_NAME = f"NHLA_1Side_{RUN_TIMESTAMP}"
LOG_NAME = f"NHLA_1Side_{RUN_TIMESTAMP}"
METRICS_RUN_NAME = f"nhla_1side_{RUN_TIMESTAMP}"

# Model config
ENCODER_NAME = "resnet18"  # options: resnet18, resnet34, resnet50
PRETRAINED = True

print(f"CSV path: {CSV_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"TensorBoard dir: {TENSORBOARD_DIR}")
print(f"Metrics dir: {METRICS_DIR}")
print(f"Image scale: {IMAGE_SCALE}, canvas: {CANVAS}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL}")
print(f"Checkpoint name: {CHECKPOINT_NAME}")
print(f"Log name: {LOG_NAME}")
print(f"Metrics run name: {METRICS_RUN_NAME}")
print("Optuna search space:")
print(json.dumps(OPTUNA_SEARCH_SPACE, indent=2, default=str))

In [ ]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import notebooks.NHLA_Classification as nhla_module
import notebooks.NHLA_Classification_1Side as nhla_1side_module
importlib.reload(nhla_module)
importlib.reload(nhla_1side_module)
train_1side = nhla_1side_module.train_1side

result = train_1side(
    csv_path=str(CSV_PATH),
    output_dir=str(OUTPUT_DIR),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_split=VAL_SPLIT,
    seed=SEED,
    optuna_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 0,
    tune_epochs=TUNE_EPOCHS,
    encoder_name=ENCODER_NAME,
    pretrained=PRETRAINED,
    search_space=OPTUNA_SEARCH_SPACE,
    study_name=STUDY_NAME,
    db_path=str(OPTUNA_DB_PATH),
    checkpoint_name=CHECKPOINT_NAME,
    log_name=LOG_NAME,
    metrics_run_name=METRICS_RUN_NAME,
    tensorboard_dir=str(TENSORBOARD_DIR),
    metrics_dir=str(METRICS_DIR),
    checkpoint_interval=CHECKPOINT_INTERVAL,
    num_workers=NUM_WORKERS,
    tune_subset_fraction=TUNE_SUBSET_FRACTION,
    image_cache_dir=IMAGE_CACHE_DIR,
    image_scale=IMAGE_SCALE,
    canvas=CANVAS,
)

print(json.dumps({
    "best_params": result["best_params"],
    "metrics": result["result"]["metrics"],
    "metrics_path": result["result"].get("metrics_path"),
}, indent=2))

## Notes

- Optuna is run before the final training step.
- The tuned parameters are saved to `best_params.json` in the output directory.
- Final training uses the best parameters rather than the initial defaults.